In [ ]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/mimi-to-hubert-bridge.git

In [ ]:
!pip install -q moshi safetensors huggingface_hub onnxruntime-gpu librosa tqdm

In [ ]:
# !apt update && apt install -y unzip
!apt-get update -qq && apt-get install -y aria2 unzip zstd

In [ ]:
#_aKHzCnOPrapIEwfnuxyQUszisfBqaRrTXQ

In [ ]:
from huggingface_hub import snapshot_download
import os
from huggingface_hub import login

# Paste your token here
hf_token = ""
login(token=hf_token)


repo_id = "Darknsu/librispeech-dataset-small-part"  # make sure this is actually a dataset repo
download_dir = "/workspace/mimi-to-hubert-bridge"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading entire dataset...")

# Download full dataset repo
snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",   # ✅ changed from "model" to "dataset"
    local_dir=download_dir,
    local_dir_use_symlinks=False
)

print("✅ All files downloaded to:", download_dir)

In [ ]:
%cd mimi-to-hubert-bridge

In [ ]:
!tar -I zstd -xf librispeech_10000_audio.tar.zst

In [ ]:
# hubert_streaming_fix_kv.onnx model download    

import os
from huggingface_hub import hf_hub_download

# Define target directory
base_dir = "/workspace/mimi-to-hubert-bridge"
model_dir = os.path.join(base_dir, "model")

# Create directories if they don't exist
os.makedirs(model_dir, exist_ok=True)

# Download the file into the model folder
file_path = hf_hub_download(
    repo_id="digital-avatar/ditto-talkinghead",
    filename="ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx",
    local_dir=model_dir,
    local_dir_use_symlinks=False
)

print("Downloaded to:", file_path)

In [ ]:
import shutil
import os

src = "/workspace/mimi-to-hubert-bridge/model/ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx"
dst = "/workspace/mimi-to-hubert-bridge/model/hubert_streaming_fix_kv.onnx"

# Check if source file exists
if os.path.exists(src):
    shutil.move(src, dst)
    print("✅ File moved successfully!")
else:
    print("❌ Source file not found!")

In [ ]:
!pip install -r requirements.txt

In [ ]:
!torchrun --nproc_per_node=5 preprocess.py \
    --dataset librispeech \
    --root ./data/LibriSpeech/train-clean-100 \
    --out_dir data \
    --val_frac 0.1 \
    --preextract \
    --device cuda \
    --num_workers 16

In [ ]:
!tar -I 'zstd -1' -cf librispeech_10000_audio_preprocess.tar.zst ./data

In [ ]:
!torchrun --nproc_per_node=5 train.py --config config.yaml